# e-SNLI — Gemma3-27b-it Few-Shot CoT: Activation Extraction & SAE Encoding

This notebook:
1. Streams the e-SNLI dataset from HuggingFace (500 samples).
2. Builds **few-shot Chain-of-Thought** prompts using `ESNLI_Dataset`.
3. Generates model responses and captures **residual-stream activations**.
4. Deletes the model to reclaim GPU memory.
5. Loads a JumpReLU SAE and encodes activations to **sparse tensors**.
6. Saves prompts, activations, and sparse SAE features to disk.

In [1]:
import sys, os, gc, pathlib

sys.path.insert(0, os.path.dirname(os.getcwd()))

import torch
from tqdm import tqdm
from datasets import load_dataset as hf_load_dataset, Dataset as HFDataset

from src.configs import ModelConfig, InferenceConfig, PromptStyle, SAEConfig, DatasetConfig
from src.dataset.esnli import ESNLI_Dataset, _LABEL_MAP_
from src.gemma_model import GemmaModel
from src.SAE import JumpReLUSAE

/workspace/LASR-main/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## HF Authentication

In [2]:
from huggingface_hub import login
from dotenv import load_dotenv

load_dotenv()
hf_token = os.getenv("HF_TOKEN")
if hf_token:
    login(token=hf_token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## Configuration

In [3]:
NUM_SAMPLES = 500
BATCH_SIZE = 2
MAX_NEW_TOKENS = 256
SAE_LAYER = 40

model_config = ModelConfig(model_name="google/gemma-3-27b-it", torch_dtype=torch.bfloat16)
inference_config = InferenceConfig(batch_size=BATCH_SIZE, max_new_tokens=MAX_NEW_TOKENS)

sae_config = SAEConfig(
    repo_id="google/gemma-scope-2-27b-it",
    sae_type="resid_post",
    layer=SAE_LAYER,
    width="65k",
    l0="medium",
)

print(f"Model:      {model_config.model_name}")
print(f"Device:     {model_config.device}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Samples:    {NUM_SAMPLES}")
print(f"Prompt:     chain_of_thought_tags (few-shot)")
print(f"SAE:        layer {SAE_LAYER}, width {sae_config.width}, l0 {sae_config.l0}")

Model:      google/gemma-3-27b-it
Device:     cuda
Batch size: 2
Samples:    500
Prompt:     chain_of_thought_tags (few-shot)
SAE:        layer 40, width 65k, l0 medium


## Load e-SNLI via Streaming

Use `streaming=True` to avoid downloading the full dataset into memory.
We materialize only the first 500 rows into a regular HF `Dataset`.

In [4]:
stream = hf_load_dataset(
    "esnli/esnli",
    split="validation",
    streaming=True,
    revision="refs/convert/parquet",
)

rows = []
for i, row in enumerate(stream):
    if i >= NUM_SAMPLES:
        break
    row["gold_label"] = _LABEL_MAP_[row["label"]]
    rows.append(row)

data = HFDataset.from_list(rows)
print(f"Materialized {len(data)} samples from streaming")
print(f"Columns: {data.column_names}")
data.to_pandas().head(3)

Materialized 500 samples from streaming
Columns: ['premise', 'hypothesis', 'label', 'explanation_1', 'explanation_2', 'explanation_3', 'gold_label']


,premise,hypothesis,label,explanation_1,explanation_2,explanation_3,gold_label
0,Two women are embracing while holding to go pa...,The sisters are hugging goodbye while holding ...,1,The to go packages may not be from lunch.,"Just because two women are embracing, does not...",Two women do not have to be sisters. Embracin...,neutral
1,Two women are embracing while holding to go pa...,Two woman are holding packages.,0,Saying the two women are holding packages is a...,Sentence 1 states that two women are holding t...,Women can embrace while they are holding packa...,entailment
2,Two women are embracing while holding to go pa...,The men are fighting outside a deli.,2,In the first sentence there is an action of af...,Women are different than men and embracing is ...,First sentence features two women and the seco...,contradiction


## Build Few-Shot CoT Prompts

We subclass `ESNLI_Dataset` to inject the pre-loaded streamed data
instead of re-downloading, then call `build_prompts()` which internally
uses `_build_few_shot_examples()` to prepend one example per label class.

In [5]:
class StreamedESNLI(ESNLI_Dataset):
    """Thin wrapper that accepts pre-loaded data to skip re-downloading."""
    def __init__(self, config, preloaded_data):
        self.path = config.path
        self.prompt_style = config.prompt_style
        self.use_chat_template = config.use_chat_template
        self.few_shot = config.few_shot
        self.data = preloaded_data

dataset_config = DatasetConfig(
    path="esnli/esnli",
    prompt_style=PromptStyle.CHAIN_OF_THOUGHT_TAGS,
    use_chat_template=False,
    few_shot=True,
)

esnli_dataset = StreamedESNLI(dataset_config, data)
prompted_data = esnli_dataset.build_prompts()

prompts_list = prompted_data["prompt"]
gold_labels = prompted_data["gold_label"]

print(f"Built {len(prompts_list)} few-shot CoT prompts")
print(f"\n--- First prompt (truncated) ---")
print(prompts_list[0][:600])
print("...")

Building prompts: 100%|██████████| 500/500 [00:00<00:00, 14042.05 examples/s]

Built 500 few-shot CoT prompts

--- First prompt (truncated) ---
<start_of_turn>user Task: Determine the logical relationship between a Premise and a Hypothesis.
Options: entailment, contradiction, neutral.

Rules:
1. You MUST provide your reasoning inside <reasoning> tags.
2. You MUST provide the final label inside <label> tags.
3. The reasoning must come BEFORE the label.

Example 1:
Premise: Two women are embracing while holding to go packages.
Hypothesis: The sisters are hugging goodbye while holding to go packages after just eating lunch.
<reasoning>The to go packages may not be from lunch.</reasoning>
<label>neutral</label>
Example 2:
Premise: Two wom
...


## Load Gemma3-27b Model

In [8]:
model = GemmaModel(model_config)
tokenizer = model.tokenizer
print(f"Model loaded on {model_config.device}")
print(f"GPU allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

OutOfMemoryError: CUDA out of memory. Tried to allocate 51.10 GiB. GPU 0 has a total capacity of 79.25 GiB of which 18.61 GiB is free. Process 115542 has 60.22 GiB memory in use. Including non-PyTorch memory, this process has 414.00 MiB memory in use. Of the allocated memory 0 bytes is allocated by PyTorch, and 0 bytes is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## Generate Responses & Gather Residual Activations

For each batch we:
1. Generate response tokens via `generate_batch`.
2. Run a forward pass per sample to capture the residual stream at the SAE target layer.
3. Move activations to CPU immediately to free GPU memory.

In [ ]:
all_prompts_text = []
all_residual_acts = []
all_prompt_lens = []
all_generation_texts = []

for start in tqdm(range(0, len(prompts_list), BATCH_SIZE), desc="Generating"):
    batch_prompts = prompts_list[start : start + BATCH_SIZE]

    texts, ids_list, prompt_lens = model.generate_batch(
        batch_prompts,
        max_new_tokens=MAX_NEW_TOKENS,
        batch_size=BATCH_SIZE,
    )

    for prompt_text, seq_ids, p_len, gen_text in zip(
        batch_prompts, ids_list, prompt_lens, texts
    ):
        with torch.no_grad():
            residual_acts = model.gather_residual_activations(
                sae_config.layer, seq_ids
            )
        all_residual_acts.append(residual_acts.cpu())
        all_prompts_text.append(prompt_text)
        all_prompt_lens.append(p_len)
        all_generation_texts.append(gen_text)

    torch.cuda.empty_cache()

print(f"\nCollected activations for {len(all_residual_acts)} samples")
print(f"Example activation shape: {all_residual_acts[0].shape}")
print(f"Example prompt length:    {all_prompt_lens[0]} tokens")

## Delete Model to Free GPU Memory

In [ ]:
del model
del tokenizer
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory after model deletion: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## Load SAE & Encode Activations to Sparse Tensors

The JumpReLU SAE produces highly sparse feature vectors.
We convert each output to a CSR sparse tensor for efficient storage.

In [ ]:
sae = JumpReLUSAE.from_pretrained(sae_config, device="cuda")

all_sae_sparse = []

for i, acts in enumerate(tqdm(all_residual_acts, desc="SAE encoding")):
    sae_acts = sae.encode(acts.to("cuda").float())   # (seq_len, n_features)
    sparse = sae_acts.to_sparse_csr().cpu()
    all_sae_sparse.append(sparse)

    if i == 0:
        nnz = sparse.values().numel()
        total = sae_acts.shape[0] * sae_acts.shape[1]
        print(f"Dense shape:  {sae_acts.shape}")
        print(f"Sparse type:  {sparse.layout}")
        print(f"Sparsity:     {1 - nnz / total:.4%} zeros")
        print(f"Non-zero:     {nnz:,} / {total:,}")

del sae
gc.collect()
torch.cuda.empty_cache()
print(f"\nSAE deleted. GPU memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## Save to Disk

The checkpoint contains:
- `prompts_text`: raw prompt strings (list of str)
- `generation_texts`: decoded model outputs (list of str)
- `prompt_lens`: prompt token lengths (list of int)
- `gold_labels`: ground-truth labels (list of str)
- `residual_activations`: dense activations per sample (list of Tensor `(seq, d_model)`)
- `sae_features_sparse`: CSR sparse SAE features per sample (list of sparse Tensor `(seq, n_features)`)
- `config`: dict of all configuration parameters

In [ ]:
save_dir = pathlib.Path("../data/activations")
save_dir.mkdir(parents=True, exist_ok=True)

filename = (
    f"esnli_gemma3-27b-it_cot_fewshot_{NUM_SAMPLES}samples"
    f"_layer{SAE_LAYER}_65k_resid_post.pt"
)
save_path = save_dir / filename

torch.save(
    {
        "prompts_text": all_prompts_text,
        "generation_texts": all_generation_texts,
        "prompt_lens": all_prompt_lens,
        "gold_labels": gold_labels,
        "residual_activations": all_residual_acts,
        "sae_features_sparse": all_sae_sparse,
        "config": {
            "model": model_config.model_name,
            "sae_repo": sae_config.repo_id,
            "sae_layer": sae_config.layer,
            "sae_width": sae_config.width,
            "sae_l0": sae_config.l0,
            "sae_type": sae_config.sae_type,
            "prompt_style": "chain_of_thought_tags",
            "few_shot": True,
            "num_samples": NUM_SAMPLES,
            "max_new_tokens": MAX_NEW_TOKENS,
        },
    },
    save_path,
)

size_gb = save_path.stat().st_size / 1e9
print(f"Saved to: {save_path}")
print(f"File size: {size_gb:.2f} GB")

## Verification

In [ ]:
checkpoint = torch.load(save_path, weights_only=False)

print(f"Keys:           {list(checkpoint.keys())}")
print(f"Num samples:    {len(checkpoint['prompts_text'])}")
print(f"Residual[0]:    {checkpoint['residual_activations'][0].shape}")

sp = checkpoint['sae_features_sparse'][0]
dense = sp.to_dense()
print(f"SAE[0] layout:  {sp.layout}")
print(f"SAE[0] shape:   {sp.shape}")
print(f"SAE[0] nonzero: {(dense != 0).sum().item():,} / {dense.numel():,}")
print(f"\nConfig: {checkpoint['config']}")